In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16

# Definindo os caminhos e parâmetros básicos
caminho_treino = './fer2013/train'
caminho_teste = './fer2013/test'
tamanho_lote = 32 # Quantas imagens a IA vê por vez
tamanho_imagem = (48, 48) # Tamanho padrão do FER2013

print("Carregando dados de treinamento...")
dados_treino = tf.keras.utils.image_dataset_from_directory(
    caminho_treino,
    color_mode="rgb", # A VGG16 exige imagens com 3 canais de cor (RGB)
    image_size=tamanho_imagem,
    batch_size=tamanho_lote
)

print("\nCarregando dados de validação (teste)...")
dados_teste = tf.keras.utils.image_dataset_from_directory(
    caminho_teste,
    color_mode="rgb",
    image_size=tamanho_imagem,
    batch_size=tamanho_lote
)

# Carrega a rede pré-treinada sem o "topo" (include_top=False)
base_vgg = VGG16(weights='imagenet', include_top=False, input_shape=(48, 48, 3))

# Congela os pesos para não estragar o que ela já aprendeu
base_vgg.trainable = False

# Construindo o novo "topo" da rede
# Como a base_vgg está com trainable=False, apenas as camadas abaixo aprenderão agora.
modelo_vgg = models.Sequential([
    base_vgg,           # A base da VGG16 (extrator de características)
    layers.Flatten(),   # Transforma os mapas de características 2D em um vetor 1D
    layers.Dense(256, activation='relu'), # Camada densa para processar os padrões extraídos
    layers.Dropout(0.5), # Ajuda a prevenir o overfitting durante o treinamento
    layers.Dense(7, activation='softmax')  # Saída final para as 7 classes de emoções do FER2013
])

# Compilando o modelo
# Usamos o otimizador Adam, que é padrão para esse tipo de tarefa de classificação.
modelo_vgg.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Resumo da Arquitetura
# Observe que a maioria dos parâmetros (da VGG) aparecerá como "Non-trainable params".
modelo_vgg.summary()

# Treinamento de Teste (3 épocas)
# Lembre-se: como as imagens foram carregadas em RGB, o processamento será um pouco mais lento que o baseline.
print("\nIniciando o treinamento com VGG16... Acompanhe a evolução da acurácia.")
historico = modelo_vgg.fit(
    dados_treino,
    validation_data=dados_teste,
    epochs=3
)

Carregando dados de treinamento...
Found 28709 files belonging to 7 classes.

Carregando dados de validação (teste)...
Found 7178 files belonging to 7 classes.
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 1, 1, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,847,815 (56.64 MB)

 Trainable params: 133,127 (520.03 KB)

 Non-trainable params: 14,714,688 (56.13 MB)


Iniciando o treinamento com VGG16... Acompanhe a evolução da acurácia.
Epoch 1/3
898/898 ━━━━━━━━━━━━━━━━━━━━ 76s 84ms/step - accuracy: 0.2793 - loss: 3.3647 - val_accuracy: 0.3210 - val_loss: 1.7025
Epoch 2/3
898/898 ━━━━━━━━━━━━━━━━━━━━ 78s 87ms/step - accuracy: 0.3304 - loss: 1.7108 - val_accuracy: 0.3559 - val_loss: 1.6503
Epoch 3/3
898/898 ━━━━━━━━━━━━━━━━━━━━ 81s 90ms/step - accuracy: 0.3479 - loss: 1.6505 - val_accuracy: 0.3660 - val_loss: 1.6310
